In [ ]:
# -----------------------------------------------------------------------
# PAPERMILL PARAMETERS CELL
# This cell is tagged 'parameters' so Papermill injects values here.
# Do not add any other code to this cell.
# -----------------------------------------------------------------------

# {year: {"e": float, "b": float, "d": float, "mse": float}}
params = {}

# {year: "<base64 PNG string>"}
figures_b64 = {}

# Sorted list of return-period years
years = []

# {year: {duration_min: intensity, ...}, ...}
data_dict = {}

# Optional project metadata shown on the cover page
project_name = "EBD Value Report"
project_location = ""
project_date = ""
job_number = ""
lat = ""
lon = ""

In [ ]:
import base64
import io
from IPython.display import display, Image, Markdown
import pandas as pd
from idf_analysis import minutes_to_labels, model_function
import numpy as np

In [ ]:
# Cover page
cover_lines = [f"# EBD Regression"]
if project_name:
    cover_lines.append(f"{project_name}")
if project_location:
    cover_lines.append(f"**{project_location}**")
if job_number:
    cover_lines.append(f"Project No. {job_number}")
if lat and lon:
    cover_lines.append(f"Rainfall data retrieved from NOAA Atlas-14 PFDS for coordinates: {lat}, {lon}")

display(Markdown("\n\n".join(cover_lines)))

In [ ]:
display(Markdown("## Fitted EBD Parameters"))

In [ ]:
# EBD parameter summary table
summary_rows = []
for year in years:
    p = params[str(year)]
    summary_rows.append({
        "Return Period": f"{year}-yr",
        "e": round(p['e'], 8),
        "b": round(p['b'], 8),
        "d": round(p['d'], 8),
        "MSE": round(p['mse'], 6),
    })

df_summary = pd.DataFrame(summary_rows)  # no set_index

latex_str = (
    df_summary.style
    .hide(axis='index')
    .format({
        "e":   "{:.8f}",
        "b":   "{:.8f}",
        "d":   "{:.8f}",
        "MSE": "{:.6f}",
    })
    .to_latex(hrules=True)
)

wrapped = (
    r"\begin{adjustbox}{max width=\textwidth}" + "\n" +
    latex_str + "\n" +
    r"\end{adjustbox}"
)
display(Markdown(f"```{{=latex}}\n{wrapped}\n```"))
display(Markdown("MSE Denotes mean squared error between the provided and predicted intensity values." \
"EBD Values are regressed from the input data using a non-linear least squares method. Learn more by visiting [SciPy Docs.](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)"))

In [ ]:
display(Markdown("## Input Intensity Values (in/hr)"))

In [ ]:
# Original intensity values table
first_year = years[0]
minutes = [int(k) for k in data_dict[str(first_year)].keys()]
labels = minutes_to_labels(minutes)

intensity_rows = {"Duration": labels}
for year in years:
    intensity_rows[f"{year}-yr"] = [
        round(data_dict[str(year)][str(m)], 2) for m in minutes
    ]

df_intensity = pd.DataFrame(intensity_rows)  # no set_index

# Build a formatter dict for every numeric column
numeric_cols = [c for c in df_intensity.columns if c != "Duration"]
fmt = {col: "{:.2f}" for col in numeric_cols}

latex_str = (
    df_intensity.style
    .hide(axis='index')
    .format(fmt)
    .to_latex(hrules=True)
)

wrapped = (
    r"\begin{adjustbox}{max width=\textwidth}" + "\n" +
    latex_str + "\n" +
    r"\end{adjustbox}"
)
display(Markdown(f"```{{=latex}}\n{wrapped}\n```"))

In [ ]:
display(Markdown("```{=latex}\n\\newpage\n```"))
display(Markdown("## Intensity-Duration Curves"))

In [ ]:
# One figure + params table per return period
for year in years:
    p = params[str(year)]

    display(Markdown(f"### {year}-Year Return Period"))

    # Decode base64 PNG and display natively — LaTeX exporter handles this correctly
    img_bytes = base64.b64decode(figures_b64[str(year)])
    display(Image(data=img_bytes, format='png'))

    # Small params table below each figure
    df_params = pd.DataFrame([{
        "e": round(p['e'], 8),
        "b": round(p['b'], 8),
        "d": round(p['d'], 8),
        "MSE": round(p['mse'], 6),
    }]).T
    df_params.columns = ['Value']
    display(df_params)